# Krishna's BNS/IT complaint assistant: resumable Colab training

**Normal use:** create a private Google Drive folder named `LegalAI` directly inside `MyDrive`. Upload both `Krishna_Train_Resume.ipynb` and `colab_training_input_v5.zip` into it. The shared source link is only used when this ZIP is absent.

**Continue on another person's account:** stop the source run only after a completed Drive checkpoint. Share the entire source `LegalAI` folder directly with that person's Google account as **Viewer**, with viewer download/copy allowed. Paste its folder link into `SHARED_SOURCE_FOLDER_URL`. When that account has no usable `MyDrive/LegalAI`, the notebook copies the complete shared project into a new folder owned by that account. Existing `MyDrive/LegalAI` projects continue normally and are never replaced.

Open the notebook in Colab, select **Runtime → Change runtime type → GPU**, then choose **Runtime → Run all**. Approve the Drive mount prompt and, only during a shared-folder handoff, the additional Google API authorization prompt.

The notebook trains the complaint-only LoRA first, then the optional BNS/IT Q&A comparison. It saves full checkpoints in `LegalAI/runs/` every 50 optimizer steps. If Colab disconnects, reconnect a GPU and choose **Run all** again. Completed runs are skipped; incomplete runs restart from the latest verified Drive checkpoint. The 90 sealed cases are evaluated only after both development results choose a model. The final adapter and results remain in Drive.

Google Colab can terminate without warning, so only completed checkpoints are guaranteed. This faster mode keeps the same model, data, optimizer, effective batch size and epochs as v3. If the GPU runs out of memory, it automatically retries from the last completed checkpoint using the earlier memory-saving setting.


In [ ]:
SHARED_SOURCE_FOLDER_URL = "https://drive.google.com/drive/folders/1nsyR4P7YP8thOb7iWPF84W1f5TjXnE-g?usp=sharing" # @param {type:"string"}
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/LegalAI')
print('Drive ready:', PROJECT)


In [ ]:
import subprocess, sys
REQUIREMENTS = [
    'transformers==4.57.6', 'trl==0.29.1', 'peft==0.18.1',
    'datasets==4.6.1', 'accelerate==1.12.0', 'bitsandbytes==0.49.2',
]
HANDOFF_NEEDED = (not (PROJECT / 'colab_training_input_v5.zip').is_file()
                  and bool(SHARED_SOURCE_FOLDER_URL.strip()))
if HANDOFF_NEEDED:
    REQUIREMENTS.append('google-api-python-client==2.200.0')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
import importlib.metadata as metadata
package_names = ['transformers','trl','peft','datasets','accelerate','bitsandbytes']
if HANDOFF_NEEDED:
    package_names.append('google-api-python-client')
for name in package_names:
    print(name, metadata.version(name))


In [ ]:
"""Clone a viewer-accessible LegalAI Drive folder into the current user's My Drive.

This source is embedded in the generated Colab notebook. It deliberately uses
the authenticated Drive API instead of a public downloader, so the owner can
share the source folder directly with a friend's Google account as Viewer.
"""
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import parse_qs, urlparse


DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"


def parse_drive_folder_url(url: str) -> tuple[str, str | None]:
    """Return (folder id, optional resource key) from common Drive folder URLs."""
    value = url.strip()
    match = re.search(r"/folders/([A-Za-z0-9_-]{10,})", value)
    query = parse_qs(urlparse(value).query)
    folder_id = match.group(1) if match else (query.get("id") or [""])[0]
    if not re.fullmatch(r"[A-Za-z0-9_-]{10,}", folder_id):
        raise ValueError("The shared source must be a Google Drive folder link")
    resource_key = (query.get("resourcekey") or query.get("resourceKey") or [None])[0]
    return folder_id, resource_key


def _execute(request, resource_pairs=()):
    pairs = [(item_id, key) for item_id, key in resource_pairs if key]
    if pairs:
        request.headers["X-Goog-Drive-Resource-Keys"] = ",".join(
            f"{item_id}/{key}" for item_id, key in pairs)
    return request.execute()


def _list_children(service, folder_id: str, resource_key: str | None = None) -> list[dict]:
    children = []
    page_token = None
    while True:
        request = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=("nextPageToken,files(id,name,mimeType,size,md5Checksum,"
                    "resourceKey,capabilities(canCopy))"),
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        )
        response = _execute(request, [(folder_id, resource_key)])
        children.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            return sorted(children, key=lambda item: (item["mimeType"] != DRIVE_FOLDER_MIME,
                                                       item["name"], item["id"]))


def _same_blob(source: dict, target: dict) -> bool:
    source_md5 = source.get("md5Checksum")
    target_md5 = target.get("md5Checksum")
    if source_md5 and target_md5:
        return source_md5 == target_md5 and source.get("size") == target.get("size")
    return source.get("size") is not None and source.get("size") == target.get("size")


def _copy_tree(service, source_id: str, destination_id: str,
               source_resource_key: str | None = None) -> dict[str, int]:
    counts = {"folders_created": 0, "files_copied": 0, "files_already_present": 0}
    destination_children = _list_children(service, destination_id)
    by_name: dict[str, list[dict]] = {}
    for child in destination_children:
        by_name.setdefault(child["name"], []).append(child)

    for source in _list_children(service, source_id, source_resource_key):
        matches = by_name.get(source["name"], [])
        if source["mimeType"] == DRIVE_FOLDER_MIME:
            folders = [item for item in matches if item["mimeType"] == DRIVE_FOLDER_MIME]
            if len(folders) > 1:
                raise RuntimeError(f"Destination has duplicate folders named {source['name']!r}")
            if folders:
                target = folders[0]
            else:
                target = service.files().create(
                    body={"name": source["name"], "mimeType": DRIVE_FOLDER_MIME,
                          "parents": [destination_id]}, fields="id,name,mimeType").execute()
                counts["folders_created"] += 1
            nested = _copy_tree(service, source["id"], target["id"], source.get("resourceKey"))
            for key, value in nested.items():
                counts[key] += value
            continue

        files = [item for item in matches if item["mimeType"] != DRIVE_FOLDER_MIME]
        if len(files) > 1:
            raise RuntimeError(f"Destination has duplicate files named {source['name']!r}")
        if files:
            if not _same_blob(source, files[0]):
                raise RuntimeError(
                    f"Destination already contains a different file named {source['name']!r}; "
                    "rename or remove it before retrying the handoff")
            counts["files_already_present"] += 1
            continue
        if not source.get("capabilities", {}).get("canCopy", False):
            raise PermissionError(
                f"Viewer cannot copy {source['name']!r}. The owner must allow viewers to download/copy")
        request = service.files().copy(
            fileId=source["id"], body={"name": source["name"], "parents": [destination_id]},
            fields="id,name,mimeType,size,md5Checksum", supportsAllDrives=True)
        _execute(request, [(source["id"], source.get("resourceKey")),
                           (source_id, source_resource_key)])
        counts["files_copied"] += 1
    return counts


def clone_shared_folder_to_my_drive(source_url: str, destination_name: str = "LegalAI") -> dict:
    """Server-side copy a shared folder tree into the authenticated user's My Drive."""
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build

    source_id, resource_key = parse_drive_folder_url(source_url)
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    request = service.files().get(
        fileId=source_id, fields="id,name,mimeType,resourceKey,capabilities(canCopy)",
        supportsAllDrives=True)
    source = _execute(request, [(source_id, resource_key)])
    if source.get("mimeType") != DRIVE_FOLDER_MIME:
        raise ValueError("The shared source link does not point to a Drive folder")

    escaped = destination_name.replace("'", "\\'")
    response = service.files().list(
        q=(f"name = '{escaped}' and 'root' in parents and trashed = false and "
           f"mimeType = '{DRIVE_FOLDER_MIME}'"),
        fields="files(id,name,mimeType)", pageSize=10).execute()
    destinations = response.get("files", [])
    if len(destinations) > 1:
        raise RuntimeError(f"My Drive contains multiple folders named {destination_name!r}")
    if destinations:
        destination = destinations[0]
    else:
        destination = service.files().create(
            body={"name": destination_name, "mimeType": DRIVE_FOLDER_MIME,
                  "parents": ["root"]}, fields="id,name,mimeType").execute()
    if destination["id"] == source_id:
        raise ValueError("The source folder and destination folder are the same")
    counts = _copy_tree(service, source_id, destination["id"],
                        source.get("resourceKey") or resource_key)
    return {"source_id": source_id, "destination_id": destination["id"], **counts}


def ensure_legal_ai_folder(project: Path, required_filename: str,
                           shared_source_folder_url: str) -> None:
    """Keep normal mounted-Drive behavior, cloning only when required input is absent."""
    required = project / required_filename
    if required.is_file():
        print("Using existing Drive project:", project)
        return
    if not shared_source_folder_url.strip():
        raise FileNotFoundError(
            f"{required} is missing. Upload it normally or provide SHARED_SOURCE_FOLDER_URL")
    print("LegalAI training input is absent in this account; copying the shared project...")
    result = clone_shared_folder_to_my_drive(shared_source_folder_url, project.name)
    print("Shared project copy complete:", result)
    for _ in range(30):
        if required.is_file():
            print("Copied project is visible in mounted Drive:", project)
            return
        time.sleep(2)
    raise RuntimeError(
        "Drive finished the server-side copy but the mounted folder has not refreshed yet. "
        "Run all once more; the copied LegalAI folder will be reused without copying again.")


ensure_legal_ai_folder(PROJECT, 'colab_training_input_v5.zip', SHARED_SOURCE_FOLDER_URL)
INPUT_ZIP = PROJECT / 'colab_training_input_v5.zip'


In [ ]:
import hashlib, json, shutil, zipfile
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Select Runtime > Change runtime type > GPU, then Run all again.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
LOCAL = Path('/content/legal_ai_input')
LOCAL.mkdir(parents=True, exist_ok=True)
LOCAL_ZIP = LOCAL / INPUT_ZIP.name
shutil.copyfile(INPUT_ZIP, LOCAL_ZIP)
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    if archive.testzip() is not None:
        raise ValueError('Dataset ZIP failed CRC check')
    for name in archive.namelist():
        if name.startswith('/') or '..' in Path(name).parts:
            raise ValueError('Unsafe ZIP entry: ' + name)
    archive.extractall(LOCAL)
INPUT = LOCAL / 'colab_training_input_v5'
manifest = json.loads((INPUT / 'manifest.json').read_text(encoding='utf-8'))
for rel, expected in manifest['files_sha256'].items():
    actual = hashlib.sha256((INPUT / rel).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError('Dataset file hash mismatch: ' + rel)
print('Frozen dataset verified:', manifest['counts'])


In [ ]:
import sys
sys.path.insert(0, str(INPUT))
sys.modules.pop('train_runner', None)  # Refresh code if this Colab session ran an older ZIP.
import train_runner
results = train_runner.run(INPUT, PROJECT)
print('Training and evaluation finished. Files are in:', PROJECT)


In [ ]:
# ============================================================
# AUTOMATIC 21-COMPLAINT BATCH TEST
# ============================================================

import os
import json
import torch
from pathlib import Path
from IPython.display import display, Markdown, JSON

# Change this to True only when you intentionally want to
# delete the previous batch results and test everything again.
RESTART_BATCH = False

RESULTS_PATH = PROJECT / "manual_batch_test_results_v1.json"
TEST_WORK = Path("/content/legal_ai_batch_test")
TEST_WORK.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "Read the reported incident. Return JSON only with outcome, "
    "possible_provision_ids, and uncertain_sections. "
    "A section is only a possibility, never a finding of guilt. "
    "Never invent missing facts."
)

TEST_CASES = [
    {
        "case_id": "TEST_001",
        "category": "clear_theft",
        "review_goal": "Check whether the model recognises an apparent theft.",
        "complaint": (
            "I was travelling on a crowded bus when someone opened my "
            "backpack and removed my phone without my permission. I noticed "
            "it missing immediately after getting off the bus."
        )
    },
    {
        "case_id": "TEST_002",
        "category": "physical_injury",
        "review_goal": "Check injury and grievous-hurt distinction.",
        "complaint": (
            "During an argument, my neighbour intentionally struck my arm "
            "with an iron rod. My arm was fractured, and I have hospital "
            "records and an X-ray."
        )
    },
    {
        "case_id": "TEST_003",
        "category": "sexual_harassment",
        "review_goal": "Check recognition of sexual harassment facts.",
        "complaint": (
            "My manager repeatedly made sexual remarks about my body and "
            "asked me for sexual favours. I clearly told him to stop, and "
            "two coworkers heard him."
        )
    },
    {
        "case_id": "TEST_004",
        "category": "stalking",
        "review_goal": "Check repeated following and unwanted contact.",
        "complaint": (
            "I told a man several times that I did not want any contact "
            "with him. He continues following me to work, waiting outside "
            "my house and monitoring my social-media activity."
        )
    },
    {
        "case_id": "TEST_005",
        "category": "voyeurism_privacy",
        "review_goal": "Check physical and electronic privacy provisions.",
        "complaint": (
            "Someone secretly recorded me while I was changing clothes in "
            "a private changing room. I never consented to the recording, "
            "and he later sent the video to another person."
        )
    },
    {
        "case_id": "TEST_006",
        "category": "bank_impersonation",
        "review_goal": "Check cyber impersonation, identity misuse and cheating.",
        "complaint": (
            "A person called me pretending to be an employee of my bank. "
            "He convinced me to provide an OTP and used it to transfer "
            "₹45,000 from my account."
        )
    },
    {
        "case_id": "TEST_007",
        "category": "fake_social_identity",
        "review_goal": "Check online identity impersonation.",
        "complaint": (
            "Someone created an Instagram account using my name and "
            "photograph. They are pretending to be me and asking my friends "
            "to send money."
        )
    },
    {
        "case_id": "TEST_008",
        "category": "account_intrusion",
        "review_goal": "Check unauthorised account access.",
        "complaint": (
            "Someone accessed my email account without permission, changed "
            "the password and downloaded private documents stored in it. "
            "The login history shows an unfamiliar device."
        )
    },
    {
        "case_id": "TEST_009",
        "category": "intimate_image_distribution",
        "review_goal": "Check non-consensual electronic distribution.",
        "complaint": (
            "My former partner shared my private intimate photographs in a "
            "WhatsApp group without my consent. I have screenshots showing "
            "who posted them."
        )
    },
    {
        "case_id": "TEST_010",
        "category": "online_shopping_fraud",
        "review_goal": "Check dishonest intention and online cheating.",
        "complaint": (
            "I saw an advertisement for a laptop and transferred ₹35,000 "
            "to the seller. After receiving the payment, the seller blocked "
            "me, deleted the advertisement and never delivered the laptop."
        )
    },
    {
        "case_id": "TEST_011",
        "category": "missing_information",
        "review_goal": "The model should avoid assuming that a lost phone was stolen.",
        "complaint": (
            "I cannot find my phone after attending a crowded wedding. "
            "I do not know whether someone stole it or whether I accidentally "
            "left it somewhere."
        )
    },
    {
        "case_id": "TEST_012",
        "category": "missing_information",
        "review_goal": "The model should distinguish delay from proven fraud.",
        "complaint": (
            "I paid someone for a product, but it has not arrived yet. "
            "The seller says delivery is delayed. I do not know whether "
            "the seller intended to deceive me."
        )
    },
    {
        "case_id": "TEST_013",
        "category": "missing_information",
        "review_goal": "The model should not assume hacking without supporting facts.",
        "complaint": (
            "My social-media password stopped working. I think someone may "
            "have hacked my account, but I have no login alert or other evidence."
        )
    },
    {
        "case_id": "TEST_014",
        "category": "missing_information",
        "review_goal": "The model should request or identify missing assault facts.",
        "complaint": (
            "A person injured me yesterday. I have not explained how it "
            "happened, whether it was intentional, what injury occurred or "
            "whether a weapon was used."
        )
    },
    {
        "case_id": "TEST_015",
        "category": "entrusted_property",
        "review_goal": "Check misuse of property that was voluntarily entrusted.",
        "complaint": (
            "I gave my laptop to a repair-shop owner only for repairs. "
            "Instead of returning it, he sold it to another person and "
            "kept the money."
        )
    },
    {
        "case_id": "TEST_016",
        "category": "document_deception",
        "review_goal": "Check deception using fabricated ownership documents.",
        "complaint": (
            "A man falsely claimed that he owned a flat and showed me "
            "fabricated ownership documents. I relied on those documents "
            "and paid him ₹2 lakh as an advance."
        )
    },
    {
        "case_id": "TEST_017",
        "category": "threat_and_extortion",
        "review_goal": "Check a threat demanding money even though payment was not made.",
        "complaint": (
            "Someone sent me messages threatening to publish private "
            "photographs unless I paid ₹50,000. I did not pay, and I saved "
            "all the messages."
        )
    },
    {
        "case_id": "TEST_018",
        "category": "multiple_possible_offences",
        "review_goal": "Check identity misuse, impersonation and financial deception.",
        "complaint": (
            "A person created a fake profile using my identity, contacted "
            "my relatives while pretending to be me and convinced one "
            "relative to transfer ₹20,000."
        )
    },
    {
        "case_id": "TEST_019",
        "category": "out_of_scope",
        "review_goal": "The model should avoid creating a criminal case from an academic dispute.",
        "complaint": (
            "My university gave me fewer marks than I expected in an "
            "examination, and I want the professor arrested."
        )
    },
    {
        "case_id": "TEST_020",
        "category": "out_of_scope",
        "review_goal": "The model should recognise an ordinary service complaint.",
        "complaint": (
            "My internet connection has been slow for three days even "
            "though I paid the monthly bill."
        )
    },
    {
        "case_id": "TEST_021",
        "category": "civil_disagreement",
        "review_goal": "The model should avoid inventing criminal intent.",
        "complaint": (
            "My landlord and I disagree about deductions from my security "
            "deposit. There were no threats, forged documents or dishonest "
            "claims when the agreement was made."
        )
    }
]


# ------------------------------------------------------------
# LOAD FINAL TRAINING INFORMATION
# ------------------------------------------------------------

training_results = json.loads(
    (PROJECT / "training_results.json").read_text(encoding="utf-8")
)

CHOSEN_VARIANT = training_results["chosen"]
FINAL_RUN = PROJECT / "runs" / CHOSEN_VARIANT

training_marker = json.loads(
    (FINAL_RUN / "training_complete.json").read_text(encoding="utf-8")
)

ADAPTER_SHA256 = training_marker["adapter_sha256"]

print("Chosen adapter:", CHOSEN_VARIANT)
print("Adapter hash:", ADAPTER_SHA256)


# ------------------------------------------------------------
# LOAD VERIFIED LAW CATALOGUE
# ------------------------------------------------------------

catalogue_rows = json.loads(
    (INPUT / "evaluation" / "offence_catalogue.json").read_text(
        encoding="utf-8"
    )
)

catalogue = {
    row["provision_id"]: row
    for row in catalogue_rows
}


# ------------------------------------------------------------
# LOAD OR RESET PREVIOUS RESULTS
# ------------------------------------------------------------

if RESTART_BATCH and RESULTS_PATH.exists():
    RESULTS_PATH.unlink()
    print("Previous batch results deleted.")

saved_cases = []

if RESULTS_PATH.exists():
    existing_payload = json.loads(
        RESULTS_PATH.read_text(encoding="utf-8")
    )

    same_adapter = (
        existing_payload.get("adapter_sha256") == ADAPTER_SHA256
        and existing_payload.get("chosen_variant") == CHOSEN_VARIANT
    )

    if same_adapter:
        saved_cases = existing_payload.get("cases", [])
        print(
            f"Resuming existing batch: "
            f"{len(saved_cases)}/{len(TEST_CASES)} already completed."
        )
    else:
        raise RuntimeError(
            "The existing batch file belongs to a different adapter. "
            "Set RESTART_BATCH = True and run the cell again."
        )

completed_ids = {
    result["case_id"]
    for result in saved_cases
}

pending_cases = [
    case for case in TEST_CASES
    if case["case_id"] not in completed_ids
]


# ------------------------------------------------------------
# SAVE RESULTS SAFELY TO DRIVE
# ------------------------------------------------------------

def save_batch_results(results):
    ordered = []

    result_by_id = {
        result["case_id"]: result
        for result in results
    }

    for case in TEST_CASES:
        if case["case_id"] in result_by_id:
            ordered.append(result_by_id[case["case_id"]])

    payload = {
        "schema_version": 1,
        "test_name": "manual_21_complaint_batch_v1",
        "chosen_variant": CHOSEN_VARIANT,
        "base_model": training_results["base_model"],
        "base_revision": training_results["base_revision"],
        "adapter_sha256": ADAPTER_SHA256,
        "system_prompt": SYSTEM_PROMPT,
        "total_test_cases": len(TEST_CASES),
        "completed_test_cases": len(ordered),
        "cases": ordered
    }

    temporary_path = RESULTS_PATH.with_suffix(".json.part")

    temporary_path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8"
    )

    os.replace(temporary_path, RESULTS_PATH)

    return ordered


# ------------------------------------------------------------
# LOAD MODEL ONLY WHEN TESTS REMAIN
# ------------------------------------------------------------

if pending_cases:
    if "batch_test_model" not in globals():
        print("Loading Qwen and the selected LoRA...")
        torch.cuda.empty_cache()

        batch_test_model, batch_test_tokenizer = (
            train_runner.load_adapter(
                FINAL_RUN,
                TEST_WORK
            )
        )

        print("Model loaded.")
    else:
        print("Using the model already loaded in GPU memory.")


# ------------------------------------------------------------
# RUN ONE COMPLAINT
# ------------------------------------------------------------

def run_single_test(case):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": case["complaint"]
        }
    ]

    input_ids = batch_test_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(batch_test_model.device)

    attention_mask = torch.ones_like(
        input_ids,
        dtype=torch.long
    )

    with torch.inference_mode():
        generated = batch_test_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=batch_test_tokenizer.eos_token_id,
            eos_token_id=batch_test_tokenizer.eos_token_id
        )

    raw_output = batch_test_tokenizer.decode(
        generated[0, input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    parsed = train_runner.parse_json(raw_output)
    json_valid = isinstance(parsed, dict)

    if json_valid:
        provision_ids = parsed.get(
            "possible_provision_ids",
            []
        )

        if not isinstance(provision_ids, list):
            provision_ids = []
    else:
        provision_ids = []

    invalid_ids = [
        provision_id
        for provision_id in provision_ids
        if provision_id not in catalogue
    ]

    valid_ids = [
        provision_id
        for provision_id in provision_ids
        if provision_id in catalogue
    ]

    non_recommendable_ids = [
        provision_id
        for provision_id in valid_ids
        if not catalogue[provision_id].get(
            "recommendable",
            False
        )
    ]

    matched_sections = []

    for provision_id in valid_ids:
        section = catalogue[provision_id]

        matched_sections.append({
            "provision_id": provision_id,
            "title": section.get("title"),
            "recommendable": section.get(
                "recommendable",
                False
            ),
            "official_text_preview": section.get(
                "verified_text",
                ""
            )[:500]
        })

    return {
        "case_id": case["case_id"],
        "category": case["category"],
        "review_goal": case["review_goal"],
        "complaint": case["complaint"],
        "raw_model_output": raw_output,
        "json_valid": json_valid,
        "parsed_output": parsed,
        "outcome": (
            parsed.get("outcome")
            if json_valid else None
        ),
        "possible_provision_ids": provision_ids,
        "database_validation": {
            "all_ids_exist": len(invalid_ids) == 0,
            "invalid_provision_ids": invalid_ids,
            "non_recommendable_provision_ids":
                non_recommendable_ids,
            "matched_sections": matched_sections
        }
    }


# ------------------------------------------------------------
# RUN ALL REMAINING TESTS
# ------------------------------------------------------------

for number, case in enumerate(
    pending_cases,
    start=len(saved_cases) + 1
):
    display(Markdown(
        f"## Running {number}/{len(TEST_CASES)} — "
        f"{case['case_id']}"
    ))

    display(Markdown(
        f"**Category:** `{case['category']}`\n\n"
        f"**Complaint:** {case['complaint']}"
    ))

    try:
        result = run_single_test(case)

    except Exception as error:
        result = {
            "case_id": case["case_id"],
            "category": case["category"],
            "review_goal": case["review_goal"],
            "complaint": case["complaint"],
            "error_type": type(error).__name__,
            "error": str(error),
            "json_valid": False,
            "parsed_output": None
        }

    saved_cases.append(result)
    saved_cases = save_batch_results(saved_cases)

    display(Markdown("### Model output"))
    display(JSON(result, expanded=True))

    print(
        f"Saved {len(saved_cases)}/{len(TEST_CASES)} "
        f"to Drive.\n"
    )


# ------------------------------------------------------------
# DISPLAY FINAL SUMMARY
# ------------------------------------------------------------

saved_cases = save_batch_results(saved_cases)

print("=" * 60)
print("BATCH TEST FINISHED")
print("=" * 60)
print("Completed:", len(saved_cases), "/", len(TEST_CASES))
print("Results file:", RESULTS_PATH)

valid_json_count = sum(
    result.get("json_valid") is True
    for result in saved_cases
)

invalid_id_cases = sum(
    bool(
        result.get(
            "database_validation",
            {}
        ).get(
            "invalid_provision_ids",
            []
        )
    )
    for result in saved_cases
)

non_recommendable_cases = sum(
    bool(
        result.get(
            "database_validation",
            {}
        ).get(
            "non_recommendable_provision_ids",
            []
        )
    )
    for result in saved_cases
)

outcome_counts = {}

for result in saved_cases:
    outcome = result.get("outcome", "missing")

    outcome_counts[outcome] = (
        outcome_counts.get(outcome, 0) + 1
    )

display(Markdown("## Batch summary"))

display(JSON({
    "completed": len(saved_cases),
    "valid_json_outputs": valid_json_count,
    "cases_with_invalid_section_ids": invalid_id_cases,
    "cases_with_non_recommendable_sections":
        non_recommendable_cases,
    "outcome_counts": outcome_counts,
    "saved_file": str(RESULTS_PATH)
}, expanded=True))

display(Markdown(
    "## Next action\n\n"
    "Download or attach `manual_batch_test_results_v1.json` "
    "from your `MyDrive/LegalAI` folder and send it to me. "
    "I will review every prediction, identify mistakes and tell "
    "you exactly what should be improved."
))